# Multi-Ticker Training: HE & LE (Livestock Futures)

This notebook demonstrates multi-ticker training using the `MultiAssetMomentum` class to train a **MultiAssetWSPR** model on **HE** (Lean Hogs) and **LE** (Live Cattle) futures.

## Model Architecture: MultiAssetWSPR
The `MultiAssetWSPR` model is a 6-path architecture:
1. **Path 1 (Windowed Summary)**: LSTM over a window of summary embeddings
2. **Path 2 (Windowed Profile)**: LSTM over a window of profile embeddings  
3. **Path 3 (Recent Raster)**: Encoder processes only the most recent raster data
4. **Path 4 (Recent Sequential)**: Encoder processes only the most recent sequential data
5. **Path 5 (Recent Spatial Fusion)**: Spatially fuses the most recent profile and raster
6. **Path 6 (Meta Modality)**: MetaModalityEncoder encodes ticker identity + calendar time

## Key Features
- **Classification & Regression**: Supports both task types with proper target calculation
- **Trading-aware losses**: OrdinalCEWithAntiCollapse, CostAwareCE, DistanceWeightedCE, FocalDirectionalLoss
- **Data scaling**: `normalize_sequential_features()` and `scale_summary_data()` for consistent feature scales
- **Target calculation**: `_calculate_target_returns()` with customizable time windows and classification binning
- **Automatic schema alignment**: Summary features aligned across tickers using signature matching
- **Metadata encoding**: Ticker ID, asset class, and calendar features (month, day-of-week, day-of-year)
- **Date-based splits**: `get_loaders()` ensures no lookahead bias across tickers

## Classification Loss Functions
| Loss | Description |
|------|-------------|
| `ordinal_anti_collapse` | Distance-weighted CE + KL regularization to prevent class collapse |
| `cost_aware` | Incorporates transaction, opportunity, and direction costs |
| `distance_weighted` | Penalizes opposite-direction errors more heavily |
| `focal` | Focal loss focusing on hard examples with direction awareness |

## Expected Directory Structure
```
<DATA_ROOT>/
    HE/
        features.csv      # Summary features
        profiles.npz      # Profile arrays
        vpin.parquet      # Sequential VPIN data
        rasterized.npz    # Rasterized VPIN data
        target.csv        # Target values (will be recalculated)
        intraday.csv      # Intraday bars for target calculation
    LE/
        features.csv
        profiles.npz
        vpin.parquet
        rasterized.npz
        target.csv
        intraday.csv
```

## 1. Setup Environment

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab - skipping drive mount")

Mounted at /content/drive


In [ ]:
# Add CTAFlow to path (Colab only)
if IN_COLAB:
    import sys
    %cd /content/drive/MyDrive/CTAEnv/
    %cd CTAFlow/
    !git pull
    %cd ..

    sys.path.insert(0, '/content/drive/MyDrive/CTAEnv/CTAFlow/')
    sys.path.insert(1, '/content/drive/MyDrive/CTAEnv/SierraPy')
    !pip install -e CTAFlow -q
    !pip install optuna -q
else:
    print("Running locally - ensure CTAFlow is installed")

/content/drive/MyDrive/CTAEnv
/content/drive/MyDrive/CTAEnv/CTAFlow
Already up to date.
/content/drive/MyDrive/CTAEnv
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for CTAFlow (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 15.1 MB/s eta 0:00:00


###1B. Setup imports


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import time, timedelta

# CTAFlow imports
from CTAFlow.models.multi_asset import (
    MultiAssetMomentum,
    GenericFiles,
    SummarySelectionConfig,
)
from CTAFlow.models.deep_learning.multi_branch.dual_model import MultiAssetWSPR
from CTAFlow.data.classifications_reference import (
    get_commodities_by_category,
)

# Classification loss functions for trading
from CTAFlow.models.deep_learning.training.loss import (
    OrdinalCEWithAntiCollapse,
    CostAwareCE,
    DistanceWeightedCE,
    FocalDirectionalLoss,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Please ensure all dependencies are installed: pip install -r requirements.txt
Failed to import PyEMD module
PyTorch version: 2.9.0+cu126
CUDA available: True
CUDA device: NVIDIA L4
Using device: cuda


## 2. Configuration & Ticker Classification

In [ ]:
# --- Path Configuration ---
PROJECT_ROOT = Path.cwd().parent  # Assumes notebook is in 'CTAEnv' dir
DATA_ROOT = PROJECT_ROOT / 'features' # Root containing HE/ and LE/ subdirs
TARGET_DIR = PROJECT_ROOT / 'data' / 'targets'
RESULTS_PATH = PROJECT_ROOT / 'results' / 'multi_ticker_wspr'
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# --- Tickers to train on ---
TICKERS = ['HE', 'LE']

# --- Build ticker/asset class mappings from classifications_reference ---
# Get all livestock tickers for reference
livestock_tickers = get_commodities_by_category().get('Livestock', [])
print(f"Available livestock tickers: {livestock_tickers}")

# Create ID mappings for MetaModalityEncoder
# These must be consistent across training/inference
TICKER_ID_MAP = {t: i for i, t in enumerate(TICKERS)}
print(f"Ticker ID map: {TICKER_ID_MAP}")

# Asset class: All livestock = 0 (Livestock category)
# If training across categories, you'd use different IDs per category
ASSET_CLASS_ID_MAP = {t: 0 for t in TICKERS}  # All livestock

# Asset subclass: Could differentiate within livestock
# HE = Lean Hogs, LE = Live Cattle, GE_F = Feeder Cattle
ASSET_SUBCLASS_ID_MAP = {'HE': 0, 'LE': 1}  # Different subclasses

print(f"Asset class map: {ASSET_CLASS_ID_MAP}")
print(f"Asset subclass map: {ASSET_SUBCLASS_ID_MAP}")

Available livestock tickers: ['LE_F', 'GE_F', 'HE_F']
Ticker ID map: {'HE': 0, 'LE': 1}
Asset class map: {'HE': 0, 'LE': 0}
Asset subclass map: {'HE': 0, 'LE': 1}


In [15]:
# --- Model Hyperparameters ---
WINDOW_SIZE = 10         # Days of lookback for windowed LSTMs
D_MODEL = 128            # Embedding dimension for each encoder branch
SUM_LSTM_HIDDEN = 64     # Hidden dim for summary LSTM
PROF_LSTM_HIDDEN = 128   # Hidden dim for profile LSTM
META_HIDDEN = 64         # Hidden dim for meta modality output
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 50
VAL_SPLIT = 0.2
MAX_SEQ_LEN = 200        # Max sequence length for intraday data

# --- Task Configuration ---
TASK = 'classification'      # 'regression' or 'classification'
NUM_CLASSES = 3              # Only used if TASK='classification'

# --- Target Calculation Configuration ---
# These settings are used by _calculate_target_returns()
TARGET_TIME_END = time(13, 0)          # End time for target period (e.g., 1:00 PM)
TARGET_PERIOD_LENGTH = timedelta(minutes=60)  # Length of target period
CLF_PERCENTILES = (33, 67)             # Percentile thresholds for 3-class classification

# --- Classification Loss Selection ---
# Options: 'cross_entropy', 'ordinal_anti_collapse', 'cost_aware', 'distance_weighted', 'focal'
LOSS_TYPE = 'ordinal_anti_collapse'

# Loss-specific parameters
ORDINAL_ALPHA = 1.0           # Distance penalty strength for OrdinalCEWithAntiCollapse
ORDINAL_REG_LAMBDA = 0.05     # Anti-collapse regularization strength
COST_AWARE_TRANSACTION = 0.001  # Transaction cost for CostAwareCE (10 bps)
COST_AWARE_OPPORTUNITY = 0.5    # Opportunity cost weight
COST_AWARE_DIRECTION = 2.0      # Wrong direction penalty

print(f"Window size: {WINDOW_SIZE} days")
print(f"Task: {TASK}")
if TASK == 'classification':
    print(f"  Num classes: {NUM_CLASSES}")
    print(f"  Target time end: {TARGET_TIME_END}")
    print(f"  Target period: {TARGET_PERIOD_LENGTH}")
    print(f"  Classification percentiles: {CLF_PERCENTILES}")
    print(f"  Loss type: {LOSS_TYPE}")

Window size: 10 days
Task: classification
  Num classes: 3
  Target time end: 13:00:00
  Target period: 1:00:00
  Classification percentiles: (33, 67)
  Loss type: ordinal_anti_collapse


## 3. Initialize MultiAssetMomentum & Load Data

In [16]:
# Configure summary feature alignment
summary_config = SummarySelectionConfig(
    strategy="exact",
    prefer_windows=("240min", "120min", "60min"),
    require_substrings=("deseasonalized",),
    min_common=8,
    strict=False,
    always_include=("rv_open", "rsv_pos_open", "rsv_neg_open"),
)

# Configure expected filenames
generic_files = GenericFiles(
    summary="features.csv",
    profiles="profiles.npz",
    vpin="vpin.parquet",
    rasterized="rasterized.npz",
    target="target.csv",
    intraday="intraday.csv",
)

# Initialize the multi-asset wrapper
multi_asset = MultiAssetMomentum(
    root_dir=DATA_ROOT,
    target_dir=TARGET_DIR,
    tickers=TICKERS,
    generic_files=generic_files,
    summary_config=summary_config,
    strict=True,
)

print(f"Discovered tickers: {multi_asset.tickers}")

Discovered tickers: ['HE', 'LE']


In [17]:
# Verify file paths for each ticker
print("Checking data files for each ticker...\n")
for ticker in TICKERS:
    print(f"--- {ticker} ---")
    paths = multi_asset.paths_for_ticker(ticker, must_exist=False)
    for name, path in paths.items():
        status = "[FOUND]" if path.exists() else "[MISSING]"
        print(f"  {status} {name}: {path.name}")
    print()

Checking data files for each ticker...

--- HE ---
  [FOUND] summary: features.csv
  [FOUND] profiles: profiles.npz
  [FOUND] vpin: vpin.parquet
  [FOUND] rasterized: rasterized.npz
  [FOUND] target: target.csv
  [FOUND] intraday: intraday.csv

--- LE ---
  [FOUND] summary: features.csv
  [FOUND] profiles: profiles.npz
  [FOUND] vpin: vpin.parquet
  [FOUND] rasterized: rasterized.npz
  [FOUND] target: target.csv
  [FOUND] intraday: intraday.csv



In [18]:
# Prepare the aligned summary schema across tickers
multi_asset.prepare_summary_schema(TICKERS)

print(f"Aligned summary columns: {len(multi_asset._aligned_summary_cols or [])}")
if multi_asset._aligned_summary_cols:
    print("Sample aligned features:")
    for col in (multi_asset._aligned_summary_cols or [])[:10]:
        print(f"  - {col}")

Aligned summary columns: 55
Sample aligned features:
  - 0830_overnight_return
  - 0930_60min_deseasonalized_ask_volume
  - 0930_60min_deseasonalized_bid_volume
  - 0930_60min_deseasonalized_volume
  - 0930_60min_deseasonalized_volume_imbalance
  - 0930_60min_impact_coeff
  - 0930_60min_impact_vol
  - 0930_60min_scaled_returns
  - 0930_intraday_curve_slope_change_0830_to_0930_M1_M3
  - 0930_intraday_rel_basis_change_0830_to_0930_M1M2_vs_M2M3


In [19]:
# Load individual ticker models to inspect dimensions
print("Loading per-ticker models to inspect data dimensions...\n")

ticker_dims = {}
for ticker in TICKERS:
    model = multi_asset.get_model(ticker)

    dims = {
        'summary': model.dims.summary_dim,
        'seq': model.dims.seq_dim,
        'profile_shape': model.dims.profile_shape,
        'raster_shape': model.dims.raster_shape,
        'n_samples': len(model.target_data),
    }
    ticker_dims[ticker] = dims

    print(f"--- {ticker} ---")
    print(f"  Summary dim: {dims['summary']}")
    print(f"  Sequential dim: {dims['seq']}")
    print(f"  Profile shape: {dims['profile_shape']}")
    print(f"  Raster shape: {dims['raster_shape']}")
    print(f"  Samples: {dims['n_samples']}")
    print()

Loading per-ticker models to inspect data dimensions...

Loading /content/drive/MyDrive/features/HE/profiles.npz...
Profile Processing Complete.
  - Samples: 3682
  - Shape: torch.Size([3682, 4, 96])
--- HE ---
  Summary dim: 55
  Sequential dim: 22
  Profile shape: (4, 96)
  Raster shape: (14, 4, 128)
  Samples: 2470

Loading /content/drive/MyDrive/features/LE/profiles.npz...
Profile Processing Complete.
  - Samples: 3660
  - Shape: torch.Size([3660, 4, 96])
--- LE ---
  Summary dim: 55
  Sequential dim: 22
  Profile shape: (4, 96)
  Raster shape: (14, 4, 128)
  Samples: 2701



In [24]:
# --- Recalculate Targets and Scale Data ---
# This section:
# 1. Recalculates target values using _calculate_target_returns() for consistent classification
# 2. Applies normalize_sequential_features() to scale VPIN/orderflow data
# 3. Applies scale_summary_data() to scale summary features to basis points

print("=" * 60)
print("RECALCULATING TARGETS & SCALING DATA")
print("=" * 60)

for ticker in TICKERS:
    print(f"--- Processing {ticker} ---")
    deep_model = multi_asset.get_model(ticker)

    # 1. Recalculate target returns with classification binning
    if TASK == 'classification':
        print(f"  Recalculating targets with make_clf=True...")
        deep_model._calculate_target_returns(
            target_time_end=TARGET_TIME_END,
            period_length=TARGET_PERIOD_LENGTH,
            make_clf=True,clf_percentiles = [33,66]
        )
    else:
        print(f"  Recalculating regression targets...")
        deep_model._calculate_target_returns(
            target_time_end=TARGET_TIME_END,
            period_length=TARGET_PERIOD_LENGTH,
            make_clf=False,
        )

    # 2. Normalize sequential features (VPIN, orderflow)
    # Scales price levels to basis points relative to close
    # Scales VPIN/imbalance to comparable range
    print(f"  Normalizing sequential features...")
    deep_model.normalize_sequential_features(
        scale_to_basis_points=True,
        scale_orderflow=True,
        inplace=True,
    )

    # 3. Scale summary features
    # Applies feature-specific scaling based on name patterns
    print(f"  Scaling summary features...")
    deep_model.scale_summary_data(
        rolling_window=252,  # Use ~1 year for calendar feature z-scores
        inplace=True,
    )

    # Update dimensions after scaling
    ticker_dims[ticker]['n_samples'] = len(deep_model.target_data)
    print(f"  Final sample count: {ticker_dims[ticker]['n_samples']}")

print(  "=" * 60)
print("Data preprocessing complete!")
print("=" * 60)

RECALCULATING TARGETS & SCALING DATA
--- Processing HE ---
  Recalculating targets with make_clf=True...

TARGET CLASSIFICATION DISTRIBUTION
Threshold source: ALL DATA (potential lookahead!)
Samples used for thresholds: 288869 / 288869
------------------------------------------------------------
Class 0: 288869 samples (100.00%)
Class 1:     0 samples ( 0.00%)
Class 2:     0 samples ( 0.00%)
Percentile thresholds: [33, 66]
Threshold values: ['0.000000', '0.000000']
  Normalizing sequential features...
  Scaling summary features...
  Final sample count: 288869
--- Processing LE ---
  Recalculating targets with make_clf=True...

TARGET CLASSIFICATION DISTRIBUTION
Threshold source: ALL DATA (potential lookahead!)
Samples used for thresholds: 289405 / 289405
------------------------------------------------------------
Class 0: 289405 samples (100.00%)
Class 1:     0 samples ( 0.00%)
Class 2:     0 samples ( 0.00%)
Percentile thresholds: [33, 66]
Threshold values: ['0.000000', '0.000000']
 

In [22]:
multi_asset.get_model("LE").target_data

,0
Date,
2011-05-26,0
2011-05-26,0
2011-05-26,0
2011-05-26,0
2011-05-26,0
...,...
2026-01-02,0
2026-01-02,0
2026-01-02,0


## 4. Create WSPR Datasets with Metadata

Two approaches are shown below:
- **Option A (Recommended)**: Use `MultiAssetMomentum.get_loaders()` for automatic dataset creation with date alignment
- **Option B (Manual)**: Create `WSPRWindowDataset` manually for each ticker

The `get_loaders()` method automatically:
- Handles date-based train/val splits to prevent lookahead bias
- Injects ticker metadata (ticker_id, asset_class_id, asset_subclass_id)
- Concatenates datasets across tickers

In [ ]:
# --- Use MultiAssetMomentum.get_loaders() ---
# This method handles metadata injection for WSPR models

print("Creating DataLoaders using MultiAssetMomentum.get_loaders()...")
print(f"  use_wspr=True for WSPRWindowDataset with calendar metadata")
print(f"  val_split=True for standard time-based split\n")

train_loader, val_loader = multi_asset.get_loaders(
    tickers=TICKERS,
    mode='concat',  # Concatenate all tickers into single loaders
    # WSPR mode for MultiAssetWSPR model
    use_wspr=True,
    ticker_id_map=TICKER_ID_MAP,
    asset_class_id_map=ASSET_CLASS_ID_MAP,
    asset_subclass_id_map=ASSET_SUBCLASS_ID_MAP,
    # Use standard val_split (each ticker splits by percentage)
    val_split=True,
    val_split_size=VAL_SPLIT,
    auto_align_dates=False,  # Don't use date cutoff (simpler)
    # Windowing configuration
    windowed=True,
    window_days=WINDOW_SIZE,
    max_seq_len=MAX_SEQ_LEN,
    # DataLoader settings
    batch_size=BATCH_SIZE,
    shuffle_train=True,
    num_workers=0,
    # Spatial data
    include_spatial=True,
    use_rasterized=True,
    # Verbose for debugging
    verbose=True,
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Train samples: {len(train_loader.dataset)}")
print(f"Val samples: {len(val_loader.dataset)}")

In [ ]:
# --- Option B: Manual Dataset Creation (Alternative) ---
# Use this if you need more control over dataset creation
# NOTE: This approach requires manual handling of date alignment

"""
# Manual approach (commented out - use Option A above instead)
train_datasets = []
val_datasets = []

for ticker in TICKERS:
    print(f"\\nCreating dataset for {ticker}...")
    deep_model = multi_asset.get_model(ticker)

    # Get data from the model (already scaled)
    summary_data = deep_model.training_data['summary']
    sequential_data = deep_model.sequential_data
    profile_array = deep_model.profile_array
    profile_dates = deep_model.profile_dates
    rasterized_data = deep_model.rasterized_data
    target_data = deep_model.target_data

    # Get ticker metadata IDs
    ticker_id = TICKER_ID_MAP[ticker]
    asset_class_id = ASSET_CLASS_ID_MAP[ticker]
    asset_subclass_id = ASSET_SUBCLASS_ID_MAP[ticker]

    # Create full dataset
    full_dataset = WSPRWindowDataset(
        summary_data=summary_data,
        sequential_data=sequential_data,
        spatial_data=profile_array,
        spatial_dates=profile_dates,
        rasterized_data=rasterized_data,
        target_data=target_data,
        window_days=WINDOW_SIZE,
        max_len=MAX_SEQ_LEN,
        ticker_id=ticker_id,
        asset_class_id=asset_class_id,
        asset_subclass_id=asset_subclass_id,
        return_dates=False,
    )

    # Time-based split
    n_samples = len(full_dataset)
    n_val = int(n_samples * VAL_SPLIT)
    n_train = n_samples - n_val

    train_ds = torch.utils.data.Subset(full_dataset, list(range(n_train)))
    val_ds = torch.utils.data.Subset(full_dataset, list(range(n_train, n_samples)))

    train_datasets.append(train_ds)
    val_datasets.append(val_ds)

combined_train = ConcatDataset(train_datasets)
combined_val = ConcatDataset(val_datasets)

train_loader = DataLoader(combined_train, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=wspr_collate_fn, num_workers=0, drop_last=True)
val_loader = DataLoader(combined_val, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=wspr_collate_fn, num_workers=0, drop_last=False)
"""
print("Option B (manual approach) is commented out - using Option A with get_loaders() instead")

In [ ]:
# Inspect a sample batch to verify shapes and meta dict
sample_batch = next(iter(train_loader))
summary_days, profile_days, raster_recent, seq_recent, seq_lens, targets, meta = sample_batch

print("Sample batch shapes:")
print(f"  summary_days:  {summary_days.shape}  (B, W, F_sum)")
print(f"  profile_days:  {profile_days.shape}  (B, W, C, Bins)")
print(f"  raster_recent: {raster_recent.shape}  (B, T, C, Bins)")
print(f"  seq_recent:    {seq_recent.shape}  (B, max_len, F_seq)")
print(f"  seq_lens:      {seq_lens.shape}  (B,)")
print(f"  targets:       {targets.shape}  (B,)")
print(f"\nMeta dict keys: {list(meta.keys())}")
print(f"  ticker_id:        {meta['ticker_id'].shape} -> {meta['ticker_id'][:4]}")
print(f"  asset_class_id:   {meta['asset_class_id'].shape}")
print(f"  asset_subclass_id:{meta['asset_subclass_id'].shape}")
print(f"  month:            {meta['month'].shape}  (B, W)")
print(f"  dow:              {meta['dow'].shape}  (B, W)")
print(f"  doy_sin:          {meta['doy_sin'].shape}  (B, W)")
print(f"  doy_cos:          {meta['doy_cos'].shape}  (B, W)")

## 5. Initialize MultiAssetWSPR Model

In [ ]:
# Get dimensions from the first ticker (should be aligned across all)
first_ticker = TICKERS[0]
first_model = multi_asset.get_model(first_ticker)

SUMMARY_DIM = first_model.dims.summary_dim
SEQ_DIM = first_model.dims.seq_dim - 1
PROFILE_CHANNELS, PROFILE_BINS = first_model.dims.profile_shape
RASTER_BARS, RASTER_CHANNELS, RASTER_BINS = first_model.dims.raster_shape

print(f"Model input dimensions:")
print(f"  Summary: {SUMMARY_DIM}")
print(f"  Sequential: {SEQ_DIM}")
print(f"  Profile: ({PROFILE_CHANNELS}, {PROFILE_BINS})")
print(f"  Raster: ({RASTER_BARS}, {RASTER_CHANNELS}, {RASTER_BINS})")

In [ ]:
# Initialize the MultiAssetWSPR model
# This model has 6 paths including the meta modality for ticker/time encoding

model = MultiAssetWSPR(
    f_sum=SUMMARY_DIM,
    f_profile=PROFILE_CHANNELS,
    f_raster=RASTER_CHANNELS,
    f_seq=SEQ_DIM,
    d_model=D_MODEL,
    sum_lstm_hidden=SUM_LSTM_HIDDEN,
    prof_lstm_hidden=PROF_LSTM_HIDDEN,
    meta_hidden=META_HIDDEN,
    # MetaModalityEncoder dimensions
    n_tickers=len(TICKERS),
    n_asset_classes=len(set(ASSET_CLASS_ID_MAP.values())),
    n_asset_subclasses=len(set(ASSET_SUBCLASS_ID_MAP.values())),
    task=TASK,
    num_classes=NUM_CLASSES if TASK == 'classification' else 1,
    dropout=0.3,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")
print(f"Model architecture:")
print(f"  - Path 1: Windowed Summary LSTM (hidden={SUM_LSTM_HIDDEN})")
print(f"  - Path 2: Windowed Profile LSTM (hidden={PROF_LSTM_HIDDEN})")
print(f"  - Path 3: Recent Raster Encoder (d_model={D_MODEL})")
print(f"  - Path 4: Recent Sequential Encoder (d_model={D_MODEL})")
print(f"  - Path 5: Recent Spatial Fusion (gated)")
print(f"  - Path 6: Meta Modality (tickers={len(TICKERS)}, hidden={META_HIDDEN})")

In [ ]:
# Define loss and optimizer
from CTAFlow.models.deep_learning.training.loss import ProfitAwareCE
if TASK == 'classification':
    # Select loss function based on LOSS_TYPE configuration
    if LOSS_TYPE == 'ordinal_anti_collapse':
        # Ordinal CE with distance weighting + anti-collapse regularization
        # Prevents model from collapsing to single class prediction
        criterion = OrdinalCEWithAntiCollapse(
            alpha=ORDINAL_ALPHA,           # Distance penalty strength (1.0 = moderate)
            reg_lambda=ORDINAL_REG_LAMBDA,  # Anti-collapse strength (0.05 = mild)
            target_probs=None,              # None = uniform [1/3, 1/3, 1/3]
            reduction='mean',
        ).to(device)  # Move to device for buffer tensors
        print(f"Using OrdinalCEWithAntiCollapse (alpha={ORDINAL_ALPHA}, lambda={ORDINAL_REG_LAMBDA})")

    elif LOSS_TYPE == 'cost_aware':
        # Incorporates trading transaction costs and opportunity costs
        criterion = CostAwareCE(
            transaction_cost=COST_AWARE_TRANSACTION,
            opportunity_cost=COST_AWARE_OPPORTUNITY,
            direction_cost=COST_AWARE_DIRECTION,
            reduction='mean',
        ).to(device)
        print(f"Using CostAwareCE (txn={COST_AWARE_TRANSACTION}, opp={COST_AWARE_OPPORTUNITY}, dir={COST_AWARE_DIRECTION})")

    elif LOSS_TYPE == 'distance_weighted':
        # Penalizes opposite direction errors more than adjacent errors
        criterion = DistanceWeightedCE(alpha=1.0, reduction='mean').to(device)
        print("Using DistanceWeightedCE (alpha=1.0)")

    elif LOSS_TYPE == 'focal':
        # Focal loss with directional awareness
        criterion = FocalDirectionalLoss(gamma=2.0, direction_gamma=1.0, reduction='mean').to(device)
        print("Using FocalDirectionalLoss (gamma=2.0, direction_gamma=1.0)")

    else:
        # Default: Standard CrossEntropyLoss
        criterion = nn.CrossEntropyLoss().to(device)
        print("Using CrossEntropyLoss (standard)")

else:
    # Regression task
    criterion = nn.MSELoss().to(device)
    print("Using MSELoss for regression")

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=LEARNING_RATE * 0.01)

print(f"\nOptimizer: AdamW (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"Scheduler: CosineAnnealingLR (T_max={NUM_EPOCHS})")

## 6. Training Loop

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_metric': [], 'val_metric': []}
best_val_loss = float('inf')
best_model_state = None

print(f"Training MultiAssetWSPR on {len(TICKERS)} tickers: {TICKERS}")
print(f"Task: {TASK}")
if TASK == 'classification':
    print(f"  Loss: {LOSS_TYPE}")
    print(f"  Classes: {NUM_CLASSES}")
print(f"Starting training for {NUM_EPOCHS} epochs...\n")

for epoch in range(NUM_EPOCHS):
    # --- Training Phase ---
    model.train()
    train_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in train_loader:
        # Unpack batch from wspr_collate_fn
        summary_days, profile_days, raster_recent, seq_recent, seq_lens, targets, meta = batch

        # Move tensors to device
        summary_days = summary_days.to(device)
        profile_days = profile_days.to(device)
        raster_recent = raster_recent.to(device)
        seq_recent = seq_recent.to(device)
        seq_lens = seq_lens.to(device)
        targets = targets.to(device)

        # Move meta dict tensors to device
        meta = {k: v.to(device) for k, v in meta.items()}

        if TASK == 'classification':
            targets = targets.long()

        optimizer.zero_grad()

        # Forward pass through MultiAssetWSPR
        outputs = model(
            summary_days=summary_days,
            profile_days=profile_days,
            raster_recent=raster_recent,
            seq_recent=seq_recent,
            seq_lens_recent=seq_lens,
            meta=meta,
        )

        # Compute loss - don't squeeze for classification (need shape B, C)
        if TASK == 'classification':
            loss = criterion(outputs, targets)
        else:
            loss = criterion(outputs.squeeze(), targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()
        if TASK == 'classification':
            _, predicted = outputs.max(1)
            total_correct += predicted.eq(targets).sum().item()
            total_samples += targets.size(0)

    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            summary_days, profile_days, raster_recent, seq_recent, seq_lens, targets, meta = batch

            summary_days = summary_days.to(device)
            profile_days = profile_days.to(device)
            raster_recent = raster_recent.to(device)
            seq_recent = seq_recent.to(device)
            seq_lens = seq_lens.to(device)
            targets = targets.to(device)
            meta = {k: v.to(device) for k, v in meta.items()}

            if TASK == 'classification':
                targets = targets.long()

            outputs = model(
                summary_days=summary_days,
                profile_days=profile_days,
                raster_recent=raster_recent,
                seq_recent=seq_recent,
                seq_lens_recent=seq_lens,
                meta=meta,
            )

            # Compute loss - don't squeeze for classification
            if TASK == 'classification':
                loss = criterion(outputs, targets)
            else:
                loss = criterion(outputs.squeeze(), targets)

            val_loss += loss.item()

            if TASK == 'classification':
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(targets).sum().item()
                val_samples += targets.size(0)

    scheduler.step()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)

    log_msg = f"Epoch {epoch+1:02}/{NUM_EPOCHS} | Train: {avg_train_loss:.5f} | Val: {avg_val_loss:.5f}"

    if TASK == 'classification':
        train_acc = 100. * total_correct / total_samples if total_samples > 0 else 0
        val_acc = 100. * val_correct / val_samples if val_samples > 0 else 0
        history['train_metric'].append(train_acc)
        history['val_metric'].append(val_acc)
        log_msg += f" | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%"

    print(log_msg)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = model.state_dict().copy()
        print(f"  -> New best val loss: {best_val_loss:.5f}")

## 7. Save Model & Visualize Results

In [ ]:
# Save the best model with full config for reproducibility
if best_model_state:
    model_name = f"multi_asset_wspr_{'_'.join(TICKERS)}.pth"
    model_save_path = RESULTS_PATH / model_name

    torch.save({
        'model_state_dict': best_model_state,
        'tickers': TICKERS,
        'ticker_id_map': TICKER_ID_MAP,
        'asset_class_id_map': ASSET_CLASS_ID_MAP,
        'asset_subclass_id_map': ASSET_SUBCLASS_ID_MAP,
        'config': {
            # Model architecture
            'f_sum': SUMMARY_DIM,
            'f_seq': SEQ_DIM,
            'f_profile': PROFILE_CHANNELS,
            'f_raster': RASTER_CHANNELS,
            'd_model': D_MODEL,
            'sum_lstm_hidden': SUM_LSTM_HIDDEN,
            'prof_lstm_hidden': PROF_LSTM_HIDDEN,
            'meta_hidden': META_HIDDEN,
            'n_tickers': len(TICKERS),
            'n_asset_classes': len(set(ASSET_CLASS_ID_MAP.values())),
            'n_asset_subclasses': len(set(ASSET_SUBCLASS_ID_MAP.values())),
            'window_size': WINDOW_SIZE,
            # Task configuration
            'task': TASK,
            'num_classes': NUM_CLASSES if TASK == 'classification' else 1,
            # Target calculation settings (for reproducibility)
            'target_time_end': str(TARGET_TIME_END),
            'target_period_minutes': int(TARGET_PERIOD_LENGTH.total_seconds() / 60),
            'clf_percentiles': CLF_PERCENTILES if TASK == 'classification' else None,
            # Loss configuration
            'loss_type': LOSS_TYPE if TASK == 'classification' else 'mse',
        },
        'history': history,
        'best_val_loss': best_val_loss,
    }, model_save_path)

    print(f"Model saved to {model_save_path}")
    print(f"\nSaved configuration:")
    print(f"  - Task: {TASK}")
    if TASK == 'classification':
        print(f"  - Loss type: {LOSS_TYPE}")
        print(f"  - Target percentiles: {CLF_PERCENTILES}")
    print(f"  - Target time: {TARGET_TIME_END}, period: {TARGET_PERIOD_LENGTH}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'MultiAssetWSPR Training: {" + ".join(TICKERS)}', fontsize=14)

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training vs Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Metric (accuracy for classification, log loss for regression)
if TASK == 'classification' and history['train_metric']:
    axes[1].plot(history['train_metric'], label='Train Acc', linewidth=2)
    axes[1].plot(history['val_metric'], label='Val Acc', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Training vs Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].semilogy(history['train_loss'], label='Train Loss', linewidth=2)
    axes[1].semilogy(history['val_loss'], label='Val Loss', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss (log scale)')
    axes[1].set_title('Loss (Log Scale)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Best Validation Loss: {best_val_loss:.5f}")

## 8. Per-Ticker Evaluation

In [ ]:
# Evaluate the model on each ticker separately
# This helps understand if the model generalizes well across both assets

model.load_state_dict(best_model_state)
model.eval()

print("Per-ticker validation performance:\n")

# Get per-ticker loaders using mode='dict'
per_ticker_loaders = multi_asset.get_loaders(
    tickers=TICKERS,
    mode='dict',  # Returns dict[ticker -> (train_loader, val_loader)]
    use_wspr=True,
    ticker_id_map=TICKER_ID_MAP,
    asset_class_id_map=ASSET_CLASS_ID_MAP,
    asset_subclass_id_map=ASSET_SUBCLASS_ID_MAP,
    val_split=True,
    val_split_size=VAL_SPLIT,
    auto_align_dates=False,
    windowed=True,
    window_days=WINDOW_SIZE,
    max_seq_len=MAX_SEQ_LEN,
    batch_size=BATCH_SIZE,
    include_spatial=True,
    use_rasterized=True,
)

for ticker in TICKERS:
    _, ticker_val_loader = per_ticker_loaders[ticker]

    ticker_loss = 0.0
    ticker_correct = 0
    ticker_samples = 0

    with torch.no_grad():
        for batch in ticker_val_loader:
            summary_days, profile_days, raster_recent, seq_recent, seq_lens, targets, meta = batch

            summary_days = summary_days.to(device)
            profile_days = profile_days.to(device)
            raster_recent = raster_recent.to(device)
            seq_recent = seq_recent.to(device)
            seq_lens = seq_lens.to(device)
            targets = targets.to(device)
            meta = {k: v.to(device) for k, v in meta.items()}

            if TASK == 'classification':
                targets = targets.long()

            outputs = model(
                summary_days=summary_days,
                profile_days=profile_days,
                raster_recent=raster_recent,
                seq_recent=seq_recent,
                seq_lens_recent=seq_lens,
                meta=meta,
            )

            loss = criterion(outputs.squeeze() if TASK == 'regression' else outputs, targets)
            ticker_loss += loss.item()

            if TASK == 'classification':
                _, predicted = outputs.max(1)
                ticker_correct += predicted.eq(targets).sum().item()
                ticker_samples += targets.size(0)

    avg_loss = ticker_loss / len(ticker_val_loader) if len(ticker_val_loader) > 0 else 0

    if TASK == 'classification':
        acc = 100. * ticker_correct / ticker_samples if ticker_samples > 0 else 0
        print(f"{ticker}: Val Loss = {avg_loss:.5f}, Val Acc = {acc:.2f}% ({ticker_correct}/{ticker_samples})")
    else:
        print(f"{ticker}: Val Loss = {avg_loss:.5f}")

## 9. Model Inference Example

In [ ]:
# Example: How to run inference on new data
# The model checkpoint includes all necessary config to reconstruct

def load_multi_asset_wspr(checkpoint_path, device='cpu'):
    """Load a trained MultiAssetWSPR model from checkpoint."""
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config = checkpoint['config']

    model = MultiAssetWSPR(
        f_sum=config['f_sum'],
        f_profile=config['f_profile'],
        f_raster=config['f_raster'],
        f_seq=config['f_seq'],
        d_model=config['d_model'],
        sum_lstm_hidden=config['sum_lstm_hidden'],
        prof_lstm_hidden=config['prof_lstm_hidden'],
        meta_hidden=config['meta_hidden'],
        n_tickers=config['n_tickers'],
        n_asset_classes=config['n_asset_classes'],
        n_asset_subclasses=config['n_asset_subclasses'],
        task=config['task'],
        num_classes=config.get('num_classes', 3 if config['task'] == 'classification' else 1),
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()

    return model, checkpoint

print("Model loading function defined.")
print("\nTo load the trained model later:")
print("  model, checkpoint = load_multi_asset_wspr('path/to/model.pth')")
print("  ticker_id_map = checkpoint['ticker_id_map']")
print("  config = checkpoint['config']")
print("  # Then use wspr_collate_fn with WSPRWindowDataset for new data")
print("\nSaved config includes:")
print("  - task: 'classification' or 'regression'")
print("  - num_classes: number of output classes (for classification)")
print("  - target_time_end: target calculation time")
print("  - clf_percentiles: percentile thresholds (for classification)")
print("  - loss_type: loss function used during training")